In [ ]:
#互動版
!pip install ipywidgets

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

#設定痊癒變數
doors=[]
p1=None#參賽者選擇
h1=None #主持人選擇
remain=None #剩下可以換的門
game_stage='choose'

output=widgets.Output()
door_box=widgets.Output() #顯示動畫


# 按鈕區
choose_box = widgets.HBox()
decision_box = widgets.HBox()
restart_box = widgets.HBox()


def start_game():
    global doors,p1,h1,remain,game_stage
    doors = ['GIFT', 'GOAT', 'GOAT']
    random.shuffle(doors)#打亂順序
    p1=None
    h1=None
    remain=None
    game_stage='choose'
    
    with output:
        clear_output()
        print("Welcome to  Monty Hall game<33")
        print("請先選擇一扇門。")
    show_doors()

#顯示三扇門----
#CSS製作動畫

#要被翻開的門
def show_doors(revealed_doors=None):
    if revealed_doors is None:
        revealed_doors = []
    
    html = """
    <style>
    /*排列方式*/
    .door-container {
        display: flex; 
        gap: 50px;
        margin-top: 50px;
        margin-bottom: 30px;
    }

    /*門的外觀*/
    .door-card {
        width: 130px;
        height: 180px;
        perspective: 1000px;
        text-align: center;
        font-family: Arial;
    }

    /*動畫設定*/
    .door-inner {
        position: relative;
        width: 100%;
        height: 100%;
        transition: transform 1.0s;
        transform-style: preserve-3d;
    }

    /*如果doorcard有revealed就翻開*/
    .door-card.revealed .door-inner {
        transform: rotateY(180deg);
    }

    /*門的正 背面*/
    .door-front, .door-back {
        position: absolute;
        width: 100%;
        height: 100%;
        backface-visibility: hidden;
        border-radius: 12px;
        display: flex;
        justify-content: center;
        align-items: center;
        font-size: 22px;
        font-weight: bold;
        border: 3px solid #555;
    }

    /*正面*/
    .door-front {
        background: #0B1F3A;
        color: white;
    }

    /*背面*/
    .door-back {
        background: #F7F7F7;
        color: black;
        transform: rotateY(180deg);
    }

    /*參賽者選到的門:粉色光暈*/
    .selected {
        box-shadow: 0 0 15px 5px pink;
    }
    
    /*主持人打開的門:黃色光暈*/
    .opened {
        box-shadow: 0 0 15px 5px yellow;
    }
    </style>
    """
    
    html += '<div class="door-container">'
    
    for i in range(3):
        revealed_class = "revealed" if i in revealed_doors else ""
        selected_class = "selected" if i == p1 else ""
        opened_class = "opened" if i == h1 else ""
        
        if i in revealed_doors:
            back_content = doors[i]
        else:
            back_content = "?"
        
        html += f"""
        <div class="door-card {revealed_class} {selected_class} {opened_class}">
            <div class="door-inner">
                <div class="door-front">Door {i + 1}</div>
                <div class="door-back">{back_content}</div>
            </div>
        </div>
        """
    
    html += "</div>"
    
    with door_box:
        clear_output()
        display(HTML(html))


def choose_door(choice):
    global p1, h1, remain, game_stage
    
    if game_stage != "choose":
        return
    
    p1 = choice
    
    # 主持人可以打開的門：
    # 不能是參賽者選的門，而且門後面要是 GOAT
    possible_open_doors = [
        i for i in range(3)
        if i != p1 and doors[i] == "GOAT"
    ]
    
    h1 = random.choice(possible_open_doors)
    
    # 剩下可以換的門
    remain = [
        i for i in range(3)
        if i != p1 and i != h1
    ][0]
    
    game_stage = "decide"
    
    # 選完門後，隱藏選門按鈕，顯示換門按鈕
    choose_box.layout.display = "none"
    decision_box.layout.display = "flex"
    
    with message_box:
        clear_output()
        print(f"主持人打開第 {h1 + 1} 扇門，裡面是 GOAT。")
        print("請選擇是否換門。")
    
    # 只翻開主持人打開的門
    show_doors(revealed_doors=[h1])


def finish_game(switch):
    global game_stage
    
    if game_stage != "decide":
        return
    
    if switch:
        final_choice = remain
    else:
        final_choice = p1
    
    game_stage = "end"
    
    # 選完後隱藏換門按鈕
    decision_box.layout.display = "none"
    
    # 最後三扇門全部翻開
    show_doors(revealed_doors=[0, 1, 2])
    
    # 最後只顯示結果
    with message_box:
        clear_output()
        
        if doors[final_choice] == "GIFT":
            print("恭喜你得到禮物！")
        else:
            print("很可惜，你得到羊。")


# 建立選門按鈕
door1_button = widgets.Button(description="選第 1 扇門")
door2_button = widgets.Button(description="選第 2 扇門")
door3_button = widgets.Button(description="選第 3 扇門")

door1_button.on_click(lambda b: choose_door(0))
door2_button.on_click(lambda b: choose_door(1))
door3_button.on_click(lambda b: choose_door(2))

choose_box.children = [door1_button, door2_button, door3_button]


# 建立換門按鈕
keep_button = widgets.Button(description="不換門")
switch_button = widgets.Button(description="換門")

keep_button.on_click(lambda b: finish_game(False))
switch_button.on_click(lambda b: finish_game(True))

decision_box.children = [keep_button, switch_button]


# 建立重新開始按鈕
restart_button = widgets.Button(description="重新開始")
restart_button.on_click(lambda b: start_game())

restart_box.children = [restart_button]


# 顯示介面
display(HTML("<h3>Step 1：請選擇一扇門</h3>"))
display(choose_box)

display(door_box)

display(HTML("<h3>Step 2：是否換門</h3>"))
display(decision_box)

display(message_box)

display(HTML("<h3>重新開始</h3>"))
display(restart_box)


# 開始遊戲
start_game()